<a href="https://colab.research.google.com/github/paru7676/Document--chatbot--project/blob/main/pdfchatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers faiss-cpu PyPDF2 gradio
!pip install -q langchain



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 18.3 MB/s eta 0:00:00


In [ ]:
import os
import glob
from pathlib import Path
import PyPDF2
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import textwrap

from transformers import pipeline


In [ ]:
DOC_FOLDER = "/content/docs"
os.makedirs(DOC_FOLDER, exist_ok=True)

In [ ]:
def extract_text_from_pdf(path):
    text = []
    with open(path, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text.append(page_text)
    return "\n".join(text)

documents = []
for filepath in glob.glob(os.path.join(DOC_FOLDER, "*")):
    name = os.path.basename(filepath)
    if filepath.lower().endswith(".pdf"):
        txt = extract_text_from_pdf(filepath)
    elif filepath.lower().endswith(".txt"):
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            txt = f.read()
    else:
        continue
    documents.append((name, txt))

print(f"Loaded {len(documents)} documents.")
if documents:
    print(documents[0][0])
    print(textwrap.shorten(documents[0][1], width=500))


Loaded 1 documents.
Aarush_Impact.pdf
Aarush Impact Aarush Impact is a forward-thinking company focused on driving positive change through innovative solutions and sustainable practices. The company is committed to creating measurable impact across communities, industries, and the environment. Mission: To empower individuals and organizations by delivering impactful solutions that foster growth, sustainability, and social responsibility. Vision: To be a global leader in creating meaningful change, bridging the gap between [...]


In [ ]:
def chunk_text(text, max_chars=800, overlap=100):
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    chunks = []
    for p in paragraphs:
        if len(p) <= max_chars:
            chunks.append(p)
        else:

            start = 0
            while start < len(p):
                end = start + max_chars
                chunks.append(p[start:end])
                start = end - overlap
    return chunks

chunk_texts = []
chunk_meta = []
for doc_name, text in documents:
    chunks = chunk_text(text)
    for i, c in enumerate(chunks):
        chunk_texts.append(c)
        chunk_meta.append({"doc": doc_name, "chunk_id": i})
print(f"Total chunks: {len(chunk_texts)}")


Total chunks: 11


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
batch_size = 64
embeddings = []
for i in range(0, len(chunk_texts), batch_size):
    batch = chunk_texts[i:i+batch_size]
    emb = model.encode(batch, show_progress_bar=False, convert_to_numpy=True)
    embeddings.append(emb)
embeddings = np.vstack(embeddings)
print("Embeddings shape:", embeddings.shape)


Embeddings shape: (11, 384)


In [ ]:
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)
faiss.normalize_L2(embeddings)
index.add(embeddings)
print("FAISS index size:", index.ntotal)


FAISS index size: 11


In [ ]:
def get_top_k(question, k=3):
    q_emb = model.encode([question], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, k)
    results = []
    for score, idx in zip(D[0], I[0]):
        meta = chunk_meta[idx]
        results.append({"score": float(score), "text": chunk_texts[idx], "meta": meta})
    return results

if chunk_texts:
    q = "How can I volunteer?"
    res = get_top_k(q, k=3)
    for r in res:
        print(r["score"], r["meta"], r["text"][:300])


0.2982858419418335 {'doc': 'Aarush_Impact.pdf', 'chunk_id': 4} Mission: To empower individuals and organizations by delivering impactful solutions that foster
0.2803049087524414 {'doc': 'Aarush_Impact.pdf', 'chunk_id': 9} solutions - Community engagement initiatives
0.12431588768959045 {'doc': 'Aarush_Impact.pdf', 'chunk_id': 8} Key Services: - Social impact consulting - Sustainable business strategies - Technology-driven


In [ ]:
def generate_answer_simple(question, k=3):
    hits = get_top_k(question, k=k)
    answer = "\n\n".join([f"[Source: {h['meta']['doc']}] {h['text']}" for h in hits])
    return answer

print(generate_answer_simple("How can I donate to the organization?", k=3))


[Source: Aarush_Impact.pdf] Mission: To empower individuals and organizations by delivering impactful solutions that foster

[Source: Aarush_Impact.pdf] solutions - Community engagement initiatives

[Source: Aarush_Impact.pdf] Core Values: - Integrity - Innovation - Collaboration - Sustainability


In [ ]:
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

def generate_answer_qa(question, k=3):
    hits = get_top_k(question, k=k)
    context = "\n\n".join([h["text"] for h in hits])
    qa_input = {"question": question, "context": context}
    out = qa_pipeline(qa_input)
    return {"answer": out['answer'], "score": float(out['score']), "context_used": context[:1000]}


print(generate_answer_qa("How can I volunteer at X?", k=3))


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Device set to use cpu


{'answer': 'delivering impactful solutions', 'score': 0.21527719497680664, 'context_used': 'Mission: To empower individuals and organizations by delivering impactful solutions that foster\n\nsolutions - Community engagement initiatives\n\nVision: To be a global leader in creating meaningful change, bridging the gap between innovation'}


/usr/local/lib/python3.12/dist-packages/transformers/pipelines/question_answering.py:395: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(


In [ ]:
import gradio as gr
def ask_bot(question):
    if not question.strip():
        return "Please enter a question."

    try:
        results = get_top_k(question, k=3)
        if not results:
            return "Sorry, I couldn’t find anything related to that question in the documents."

        answer = ""
        for i, res in enumerate(results, start=1):
            answer += f"*From document:* {res['meta']['doc']}\n"
            answer += f"{res['text']}\n\n"
        return answer.strip()
    except Exception as e:
        return f" Error: {str(e)}"

# Gradio interface
ui = gr.Interface(
    fn=ask_bot,
    inputs=gr.Textbox(lines=2, placeholder="Ask me anything about your documents..."),
    outputs="text",
    title="Document-Based Chatbot",
    description="Upload your documents, then ask questions to get answers from their content.",
    theme="soft"
)

# Launch app (share=True gives you a public link to include in email)
ui.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e2ebb4d85f8380413b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
